# Library

In [204]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.impute import KNNImputer

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier

from scipy.stats import ttest_rel, wilcoxon

# Load Clean Dataset

In [106]:
df = pd.read_parquet(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/processed dataset/final_df_(distance).parquet"
)

final_features = [
    "total_price",
    "total_freight_value",
    "total_product_weight_g",
    "average_product_length_cm",
    "average_product_height_cm",
    "item_count",
    "max_payment_installments",
    "average_distance_km"
]

experiment_columns = [
    "order_id",
    *final_features,
    "is_delayed"
]

model_df = df[experiment_columns].dropna().copy()

print(model_df.shape)
print(model_df.isna().sum().sum())

(95960, 10)
0


In [107]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    random_state=42,
    stratify=model_df["is_delayed"]
)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (76768, 10)
Test: (19192, 10)


In [108]:
train_idx = train_df.index
test_idx = test_df.index

print("Train indices:", len(train_idx))
print("Test indices:", len(test_idx))

Train indices: 76768
Test indices: 19192


# Load Missing Dataset

In [109]:
from pathlib import Path

data_folder = Path("/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/exp_data")

injection_data = {}

for file in data_folder.glob("*.parquet"):
    injection_data[file.stem] = pd.read_parquet(file)

print(injection_data.keys())

dict_keys(['mar_50', 'mnar_50', 'mcar_30', 'mar_30', 'mnar_30', 'mcar_50', 'mcar_10', 'mar_10', 'mnar_10'])


In [122]:
mcar_10 = injection_data["mcar_10"]
mcar_30 = injection_data['mcar_30']
mcar_50 = injection_data['mcar_50']

mar_10 = injection_data['mar_10']
mar_30 = injection_data['mar_30']
mar_50 = injection_data['mar_50']

mnar_10 = injection_data['mnar_10']
mnar_30 = injection_data['mnar_30']
mnar_50 = injection_data['mnar_50']

In [123]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.2,
    random_state=42,
    stratify=model_df["is_delayed"]
)

# Function: model + results

In [ ]:
def run_experiment(
    injected_df,
    mechanism,
    missing_rate,
    method
):
    # =========================
    # 1. Restore original index
    # =========================
    injected_df = injected_df.copy()
    injected_df.index = model_df.index

    # =========================
    # 2. Train / Test Split
    # =========================
    train_df = injected_df.loc[train_idx].copy()
    test_df = injected_df.loc[test_idx].copy()

    # =========================
    # 3. Imputation
    # =========================

    if method == "Mean":

        value = train_df["total_freight_value"].mean()

        train_df["total_freight_value"] = (
            train_df["total_freight_value"].fillna(value)
        )

        test_df["total_freight_value"] = (
            test_df["total_freight_value"].fillna(value)
        )

    elif method == "Median":

        value = train_df["total_freight_value"].median()

        train_df["total_freight_value"] = (
            train_df["total_freight_value"].fillna(value)
        )

        test_df["total_freight_value"] = (
            test_df["total_freight_value"].fillna(value)
        )

    elif method in ["KNN", "MICE"]:

        # -------------------------
        # Standardize using train only
        # -------------------------
        scaler = StandardScaler()

        X_train = train_df[final_features].copy()
        X_test = test_df[final_features].copy()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # -------------------------
        # Imputer
        # -------------------------
        if method == "KNN":

            imputer = KNNImputer(
                n_neighbors=5,
                weights="uniform"
            )

        else:  # MICE

            imputer = IterativeImputer(
                max_iter=10,
                random_state=42
            )

        # -------------------------
        # Fit only on train
        # -------------------------
        X_train_imputed = imputer.fit_transform(
            X_train_scaled
        )

        # Transform test
        X_test_imputed = imputer.transform(
            X_test_scaled
        )

        # -------------------------
        # Restore original scale
        # -------------------------
        X_train_imputed = scaler.inverse_transform(
            X_train_imputed
        )

        X_test_imputed = scaler.inverse_transform(
            X_test_imputed
        )

        # -------------------------
        # Put back into DataFrames
        # -------------------------
        train_df[final_features] = X_train_imputed
        test_df[final_features] = X_test_imputed

    else:

        raise ValueError(
            f"Unknown imputation method: {method}"
        )

    # =========================
    # 4. Prepare X / y
    # =========================

    X_train = train_df[final_features]
    y_train = train_df["is_delayed"]

    X_test = test_df[final_features]
    y_test = test_df["is_delayed"]

    # =========================
    # 5. Handle Class Imbalance
    # =========================

    scale_pos_weight = (
        (y_train == 0).sum()
        / (y_train == 1).sum()
    )

    # =========================
    # 6. XGBoost
    # =========================

    xgb_model = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight
    )

    xgb_model.fit(
        X_train,
        y_train
    )

    # =========================
    # 7. Prediction
    # =========================

    y_pred = xgb_model.predict(X_test)

    y_prob = xgb_model.predict_proba(
        X_test
    )[:, 1]

    # =========================
    # 8. Evaluation
    # =========================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    # =========================
    # 9. Save Result
    # =========================

    downstream_results.append({
        "mechanism": mechanism,
        "missing_rate": missing_rate,
        "method": method,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc
    })

    # =========================
    # 10. Print Result
    # =========================

    print("=" * 50)
    print("Downstream Experiment")
    print("Model: XGBoost")
    print("Mechanism:", mechanism)
    print("Missing Rate:", f"{missing_rate:.0%}")
    print("Imputation Method:", method)
    print("-" * 50)

    print(
        "Scale Pos Weight:",
        scale_pos_weight
    )

    print(
        "Accuracy:",
        accuracy
    )

    print(
        "Precision:",
        precision
    )

    print(
        "Recall:",
        recall
    )

    print(
        "F1:",
        f1
    )

    print(
        "ROC-AUC:",
        roc_auc
    )

    print("=" * 50)

'def run_experiment(\n    injected_df,\n    mechanism,\n    missing_rate,\n    method\n):\n    # =========================\n    # 1. Restore original index\n    # =========================\n    injected_df = injected_df.copy()\n    injected_df.index = model_df.index\n\n    # =========================\n    # 2. Train / Test Split\n    # =========================\n    train_df = injected_df.loc[train_idx].copy()\n    test_df = injected_df.loc[test_idx].copy()\n\n    # =========================\n    # 3. Imputation\n    # =========================\n\n    if method == "Mean":\n\n        value = train_df["total_freight_value"].mean()\n\n        train_df["total_freight_value"] = (\n            train_df["total_freight_value"].fillna(value)\n        )\n\n        test_df["total_freight_value"] = (\n            test_df["total_freight_value"].fillna(value)\n        )\n\n    elif method == "Median":\n\n        value = train_df["total_freight_value"].median()\n\n        train_df["total_freight_valu

In [175]:
#clear the list
downstream_results = []

# MCAR

## Mean

In [176]:
run_experiment(mcar_10, "MCAR", 0.10, "Mean")
run_experiment(mcar_30, "MCAR", 0.30, "Mean")
run_experiment(mcar_50, "MCAR", 0.50, "Mean")

Downstream Experiment
Model: XGBoost
Mechanism: MCAR
Missing Rate: 10%
Imputation Method: Mean
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.649854105877449
Precision: 0.10959939531368103
Recall: 0.4662379421221865
F1: 0.17747858017135862
ROC-AUC: 0.595596311977141
Downstream Experiment
Model: XGBoost
Mechanism: MCAR
Missing Rate: 30%
Imputation Method: Mean
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6514693622342643
Precision: 0.10665032178976402
Recall: 0.447588424437299
F1: 0.1722559089221631
ROC-AUC: 0.5872882333927123
Downstream Experiment
Model: XGBoost
Mechanism: MCAR
Missing Rate: 50%
Imputation Method: Mean
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6535535639849938
Precision: 0.10911602209944751
Recall: 0.4572347266881029
F1: 0.17618634617767315
ROC-AUC: 0.5844955622561238


In [177]:
len(downstream_results)

3

## Median

In [178]:
run_experiment(mcar_10, "MCAR", 0.10, "Median")
run_experiment(mcar_30, "MCAR", 0.30, "Median")
run_experiment(mcar_50, "MCAR", 0.50, "Median")

Downstream Experiment
Model: XGBoost
Mechanism: MCAR
Missing Rate: 10%
Imputation Method: Median
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6499062109212171
Precision: 0.11102741789695691
Recall: 0.4739549839228296
F1: 0.17990967899426338
ROC-AUC: 0.5970123828031066
Downstream Experiment
Model: XGBoost
Mechanism: MCAR
Missing Rate: 30%
Imputation Method: Median
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6440704460191746
Precision: 0.10732360821673118
Recall: 0.46366559485530545
F1: 0.17430194608968935
ROC-AUC: 0.5863916237185528
Downstream Experiment
Model: XGBoost
Mechanism: MCAR
Missing Rate: 50%
Imputation Method: Median
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6564193413922468
Precision: 0.1096824167312161
Recall: 0.4553054662379421
F1: 0.17677902621722846
ROC-AUC: 0.5860326881499303


## KNN

In [179]:
run_experiment(mcar_10, "MCAR", 0.10, "KNN")
run_experiment(mcar_30, "MCAR", 0.30, "KNN")
run_experiment(mcar_50, "MCAR", 0.50, "KNN")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Downstream Experiment
Model: XGBoost
Mechanism: MCAR
Missing Rate: 10%
Imputation Method: KNN
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6414130887869945
Precision: 0.10836641670342596
Recall: 0.4739549839228296
F1: 0.17640019147917663
ROC-AUC: 0.5912065890419276


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Downstream Experiment
Model: XGBoost
Mechanism: MCAR
Missing Rate: 30%
Imputation Method: KNN
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6493851604835348
Precision: 0.10814904574371402
Recall: 0.4591639871382637
F1: 0.17506436189775654
ROC-AUC: 0.5909861193227407


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Downstream Experiment
Model: XGBoost
Mechanism: MCAR
Missing Rate: 50%
Imputation Method: KNN
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6457378074197583
Precision: 0.10595130748422002
Recall: 0.4533762057877814
F1: 0.17176269947618467
ROC-AUC: 0.588379697971252


In [180]:
downstream_results

[{'mechanism': 'MCAR',
  'missing_rate': 0.1,
  'method': 'Mean',
  'accuracy': 0.649854105877449,
  'precision': 0.10959939531368103,
  'recall': 0.4662379421221865,
  'f1': 0.17747858017135862,
  'roc_auc': 0.595596311977141},
 {'mechanism': 'MCAR',
  'missing_rate': 0.3,
  'method': 'Mean',
  'accuracy': 0.6514693622342643,
  'precision': 0.10665032178976402,
  'recall': 0.447588424437299,
  'f1': 0.1722559089221631,
  'roc_auc': 0.5872882333927123},
 {'mechanism': 'MCAR',
  'missing_rate': 0.5,
  'method': 'Mean',
  'accuracy': 0.6535535639849938,
  'precision': 0.10911602209944751,
  'recall': 0.4572347266881029,
  'f1': 0.17618634617767315,
  'roc_auc': 0.5844955622561238},
 {'mechanism': 'MCAR',
  'missing_rate': 0.1,
  'method': 'Median',
  'accuracy': 0.6499062109212171,
  'precision': 0.11102741789695691,
  'recall': 0.4739549839228296,
  'f1': 0.17990967899426338,
  'roc_auc': 0.5970123828031066},
 {'mechanism': 'MCAR',
  'missing_rate': 0.3,
  'method': 'Median',
  'accurac

## MICE

In [181]:
run_experiment(mcar_10, "MCAR", 0.10, "MICE")
run_experiment(mcar_30, "MCAR", 0.30, "MICE")
run_experiment(mcar_50, "MCAR", 0.50, "MICE")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

Downstream Experiment
Model: XGBoost
Mechanism: MCAR
Missing Rate: 10%
Imputation Method: MICE
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6496977907461442
Precision: 0.11166215810039075
Recall: 0.4778135048231511
F1: 0.18102083079546838
ROC-AUC: 0.5922685920256433


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

Downstream Experiment
Model: XGBoost
Mechanism: MCAR
Missing Rate: 30%
Imputation Method: MICE
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6546998749478949
Precision: 0.10827927092987334
Recall: 0.4508038585209003
F1: 0.17461701332669075
ROC-AUC: 0.5856742083609308


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

Downstream Experiment
Model: XGBoost
Mechanism: MCAR
Missing Rate: 50%
Imputation Method: MICE
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6537619841600667
Precision: 0.10785824345146379
Recall: 0.45016077170418006
F1: 0.1740211311373524
ROC-AUC: 0.587170077083273


In [191]:
downstream_results

[{'mechanism': 'MCAR',
  'missing_rate': 0.1,
  'method': 'Mean',
  'accuracy': 0.649854105877449,
  'precision': 0.10959939531368103,
  'recall': 0.4662379421221865,
  'f1': 0.17747858017135862,
  'roc_auc': 0.595596311977141},
 {'mechanism': 'MCAR',
  'missing_rate': 0.3,
  'method': 'Mean',
  'accuracy': 0.6514693622342643,
  'precision': 0.10665032178976402,
  'recall': 0.447588424437299,
  'f1': 0.1722559089221631,
  'roc_auc': 0.5872882333927123},
 {'mechanism': 'MCAR',
  'missing_rate': 0.5,
  'method': 'Mean',
  'accuracy': 0.6535535639849938,
  'precision': 0.10911602209944751,
  'recall': 0.4572347266881029,
  'f1': 0.17618634617767315,
  'roc_auc': 0.5844955622561238},
 {'mechanism': 'MCAR',
  'missing_rate': 0.1,
  'method': 'Median',
  'accuracy': 0.6499062109212171,
  'precision': 0.11102741789695691,
  'recall': 0.4739549839228296,
  'f1': 0.17990967899426338,
  'roc_auc': 0.5970123828031066},
 {'mechanism': 'MCAR',
  'missing_rate': 0.3,
  'method': 'Median',
  'accurac

# MAR

## Mean

In [182]:
run_experiment(mar_10, "MAR", 0.10, "Mean")
run_experiment(mar_30, "MAR", 0.30, "Mean")
run_experiment(mar_50, "MAR", 0.50, "Mean")

Downstream Experiment
Model: XGBoost
Mechanism: MAR
Missing Rate: 10%
Imputation Method: Mean
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6418820341809087
Precision: 0.10920047031158142
Recall: 0.4778135048231511
F1: 0.17777246082067233
ROC-AUC: 0.5932658196093531
Downstream Experiment
Model: XGBoost
Mechanism: MAR
Missing Rate: 30%
Imputation Method: Mean
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6511567319716548
Precision: 0.11025174400970579
Recall: 0.467524115755627
F1: 0.17842680083445822
ROC-AUC: 0.5915826619243708
Downstream Experiment
Model: XGBoost
Mechanism: MAR
Missing Rate: 50%
Imputation Method: Mean
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6495935806586077
Precision: 0.10761991499696417
Recall: 0.45594855305466236
F1: 0.17413729583691515
ROC-AUC: 0.5925841738365358


## Median

In [183]:
run_experiment(mar_10, "MAR", 0.10, "Median")
run_experiment(mar_30, "MAR", 0.30, "Median")
run_experiment(mar_50, "MAR", 0.50, "Median")

Downstream Experiment
Model: XGBoost
Mechanism: MAR
Missing Rate: 10%
Imputation Method: Median
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6417778240933723
Precision: 0.10905349794238683
Recall: 0.47717041800643084
F1: 0.17753319775092716
ROC-AUC: 0.5939462985863356
Downstream Experiment
Model: XGBoost
Mechanism: MAR
Missing Rate: 30%
Imputation Method: Median
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.648551479783243
Precision: 0.10977443609022557
Recall: 0.4694533762057878
F1: 0.17794028031687995
ROC-AUC: 0.5919312421799611
Downstream Experiment
Model: XGBoost
Mechanism: MAR
Missing Rate: 50%
Imputation Method: Median
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6488120050020842
Precision: 0.10796915167095116
Recall: 0.4591639871382637
F1: 0.17482859941234083
ROC-AUC: 0.5911563074339297


## KNN

In [184]:
run_experiment(mar_10, "MAR", 0.10, "KNN")
run_experiment(mar_30, "MAR", 0.30, "KNN")
run_experiment(mar_50, "MAR", 0.50, "KNN")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Downstream Experiment
Model: XGBoost
Mechanism: MAR
Missing Rate: 10%
Imputation Method: KNN
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6455814922884535
Precision: 0.10814040328603436
Recall: 0.46559485530546624
F1: 0.1755151515151515
ROC-AUC: 0.5948932445620478


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Downstream Experiment
Model: XGBoost
Mechanism: MAR
Missing Rate: 30%
Imputation Method: KNN
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6517298874531054
Precision: 0.10924881913758952
Recall: 0.46109324758842446
F1: 0.17664449371766444
ROC-AUC: 0.5897171376966758


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Downstream Experiment
Model: XGBoost
Mechanism: MAR
Missing Rate: 50%
Imputation Method: KNN
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6490204251771572
Precision: 0.10815307820299501
Recall: 0.45980707395498394
F1: 0.17511633602743082
ROC-AUC: 0.584269641412647


## MICE

In [185]:
run_experiment(mar_10, "MAR", 0.10, "MICE")
run_experiment(mar_30, "MAR", 0.30, "MICE")
run_experiment(mar_50, "MAR", 0.50, "MICE")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

Downstream Experiment
Model: XGBoost
Mechanism: MAR
Missing Rate: 10%
Imputation Method: MICE
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6484472696957065
Precision: 0.1118421052631579
Recall: 0.4810289389067524
F1: 0.18148732257673178
ROC-AUC: 0.5942262748930878


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

Downstream Experiment
Model: XGBoost
Mechanism: MAR
Missing Rate: 30%
Imputation Method: MICE
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6458941225510629
Precision: 0.1078856800837947
Recall: 0.46366559485530545
F1: 0.17504248604030104
ROC-AUC: 0.58716531874401


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

Downstream Experiment
Model: XGBoost
Mechanism: MAR
Missing Rate: 50%
Imputation Method: MICE
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6535535639849938
Precision: 0.10767097966728281
Recall: 0.4495176848874598
F1: 0.1737293401267553
ROC-AUC: 0.5900362016638874


# MNAR

## Mean

In [186]:
run_experiment(mnar_10, "MNAR", 0.10, "Mean")
run_experiment(mnar_30, "MNAR", 0.30, "Mean")
run_experiment(mnar_50, "MNAR", 0.50, "Mean")

Downstream Experiment
Model: XGBoost
Mechanism: MNAR
Missing Rate: 10%
Imputation Method: Mean
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6434972905377241
Precision: 0.10854435065896638
Recall: 0.47138263665594854
F1: 0.17645642753972074
ROC-AUC: 0.5922532595991291
Downstream Experiment
Model: XGBoost
Mechanism: MNAR
Missing Rate: 30%
Imputation Method: Mean
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.641517298874531
Precision: 0.10828306605855524
Recall: 0.4733118971061093
F1: 0.17624521072796934
ROC-AUC: 0.5923365214206395
Downstream Experiment
Model: XGBoost
Mechanism: MNAR
Missing Rate: 50%
Imputation Method: Mean
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6494893705710713
Precision: 0.10983705491852747
Recall: 0.4681672025723473
F1: 0.17792985457656116
ROC-AUC: 0.5860611105672141


## Median

In [187]:
run_experiment(mnar_10, "MNAR", 0.10, "Median")
run_experiment(mnar_30, "MNAR", 0.30, "Median")
run_experiment(mnar_50, "MNAR", 0.50, "Median")

Downstream Experiment
Model: XGBoost
Mechanism: MNAR
Missing Rate: 10%
Imputation Method: Median
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.644591496456857
Precision: 0.10957888493475682
Recall: 0.4752411575562701
F1: 0.1780937462344861
ROC-AUC: 0.5911164905260736
Downstream Experiment
Model: XGBoost
Mechanism: MNAR
Missing Rate: 30%
Imputation Method: Median
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.650323051271363
Precision: 0.10903851986654535
Recall: 0.46237942122186493
F1: 0.17646336973861823
ROC-AUC: 0.5877601293830732
Downstream Experiment
Model: XGBoost
Mechanism: MNAR
Missing Rate: 50%
Imputation Method: Median
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6502709462275948
Precision: 0.10890338237524648
Recall: 0.4617363344051447
F1: 0.17623956799214532
ROC-AUC: 0.5877010603439459


## KNN

In [188]:
run_experiment(mnar_10, "MNAR", 0.10, "KNN")
run_experiment(mnar_30, "MNAR", 0.30, "KNN")
run_experiment(mnar_50, "MNAR", 0.50, "KNN")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Downstream Experiment
Model: XGBoost
Mechanism: MNAR
Missing Rate: 10%
Imputation Method: KNN
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6455293872446853
Precision: 0.10824126604956703
Recall: 0.4662379421221865
F1: 0.17569368714406883
ROC-AUC: 0.5920879027519428


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Downstream Experiment
Model: XGBoost
Mechanism: MNAR
Missing Rate: 30%
Imputation Method: KNN
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6440704460191746
Precision: 0.1094166419899319
Recall: 0.4752411575562701
F1: 0.17787940787098327
ROC-AUC: 0.593120298291355


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Downstream Experiment
Model: XGBoost
Mechanism: MNAR
Missing Rate: 50%
Imputation Method: KNN
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6456335973322218
Precision: 0.10721772985923929
Recall: 0.4604501607717042
F1: 0.17393416737519737
ROC-AUC: 0.5860877827907459


## MICE

In [189]:
run_experiment(mnar_10, "MNAR", 0.10, "MICE")
run_experiment(mnar_30, "MNAR", 0.30, "MICE")
run_experiment(mnar_50, "MNAR", 0.50, "MICE")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

Downstream Experiment
Model: XGBoost
Mechanism: MNAR
Missing Rate: 10%
Imputation Method: MICE
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6451125468945393
Precision: 0.10857908847184987
Recall: 0.46881028938906755
F1: 0.1763211996613859
ROC-AUC: 0.5903787656284554


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

Downstream Experiment
Model: XGBoost
Mechanism: MNAR
Missing Rate: 30%
Imputation Method: MICE
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.6500104210087536
Precision: 0.1070341047503045
Recall: 0.45209003215434085
F1: 0.1730887603102302
ROC-AUC: 0.5860661788366206


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

Downstream Experiment
Model: XGBoost
Mechanism: MNAR
Missing Rate: 50%
Imputation Method: MICE
--------------------------------------------------
Scale Pos Weight: 11.340138241440282
Accuracy: 0.652511463109629
Precision: 0.10902140672782874
Recall: 0.4585209003215434
F1: 0.17615812229771463
ROC-AUC: 0.5851867794010217


# Results

In [190]:
downstream_results

[{'mechanism': 'MCAR',
  'missing_rate': 0.1,
  'method': 'Mean',
  'accuracy': 0.649854105877449,
  'precision': 0.10959939531368103,
  'recall': 0.4662379421221865,
  'f1': 0.17747858017135862,
  'roc_auc': 0.595596311977141},
 {'mechanism': 'MCAR',
  'missing_rate': 0.3,
  'method': 'Mean',
  'accuracy': 0.6514693622342643,
  'precision': 0.10665032178976402,
  'recall': 0.447588424437299,
  'f1': 0.1722559089221631,
  'roc_auc': 0.5872882333927123},
 {'mechanism': 'MCAR',
  'missing_rate': 0.5,
  'method': 'Mean',
  'accuracy': 0.6535535639849938,
  'precision': 0.10911602209944751,
  'recall': 0.4572347266881029,
  'f1': 0.17618634617767315,
  'roc_auc': 0.5844955622561238},
 {'mechanism': 'MCAR',
  'missing_rate': 0.1,
  'method': 'Median',
  'accuracy': 0.6499062109212171,
  'precision': 0.11102741789695691,
  'recall': 0.4739549839228296,
  'f1': 0.17990967899426338,
  'roc_auc': 0.5970123828031066},
 {'mechanism': 'MCAR',
  'missing_rate': 0.3,
  'method': 'Median',
  'accurac

In [192]:
# Convert all experiment results to DataFrame
downstream_results_df = pd.DataFrame(downstream_results)

# Check results
print(downstream_results_df.shape)
display(downstream_results_df)


(36, 8)


,mechanism,missing_rate,method,accuracy,precision,recall,f1,roc_auc
0,MCAR,0.1,Mean,0.649854,0.109599,0.466238,0.177479,0.595596
1,MCAR,0.3,Mean,0.651469,0.106650,0.447588,0.172256,0.587288
2,MCAR,0.5,Mean,0.653554,0.109116,0.457235,0.176186,0.584496
3,MCAR,0.1,Median,0.649906,0.111027,0.473955,0.179910,0.597012
4,MCAR,0.3,Median,0.644070,0.107324,0.463666,0.174302,0.586392
5,MCAR,0.5,Median,0.656419,0.109682,0.455305,0.176779,0.586033
6,MCAR,0.1,KNN,0.641413,0.108366,0.473955,0.176400,0.591207
7,MCAR,0.3,KNN,0.649385,0.108149,0.459164,0.175064,0.590986
8,MCAR,0.5,KNN,0.645738,0.105951,0.453376,0.171763,0.588380
9,MCAR,0.1,MICE,0.649698,0.111662,0.477814,0.181021,0.592269


In [193]:
# Save results
downstream_results_df.to_csv(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/downstream_results.csv",
    index=False
)

In [194]:
save_result = '/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/downstream_results.csv'

In [198]:
save_result_df = pd.read_csv(save_result)

In [199]:
save_result_df.shape

(36, 8)

In [200]:
save_result_df.head()

,mechanism,missing_rate,method,accuracy,precision,recall,f1,roc_auc
0,MCAR,0.1,Mean,0.649854,0.109599,0.466238,0.177479,0.595596
1,MCAR,0.3,Mean,0.651469,0.106650,0.447588,0.172256,0.587288
2,MCAR,0.5,Mean,0.653554,0.109116,0.457235,0.176186,0.584496
3,MCAR,0.1,Median,0.649906,0.111027,0.473955,0.179910,0.597012
4,MCAR,0.3,Median,0.644070,0.107324,0.463666,0.174302,0.586392


In [201]:
save_result_df.groupby(
    ["mechanism", "missing_rate"]
)[["accuracy", "precision", "recall", "f1", "roc_auc"]].mean()

accuracy  precision    recall        f1   roc_auc
mechanism missing_rate                                                   
MAR       0.1           0.644422   0.109559  0.475402  0.178077  0.594083
          0.3           0.649333   0.109290  0.465434  0.177014  0.590099
          0.5           0.650245   0.107853  0.456109  0.174453  0.589512
MCAR      0.1           0.647718   0.110164  0.472990  0.178702  0.594021
          0.3           0.649906   0.107601  0.455305  0.174060  0.587585
          0.5           0.652368   0.108152  0.454019  0.174687  0.586520
MNAR      0.1           0.644683   0.108736  0.470418  0.176641  0.591459
          0.3           0.646480   0.108443  0.465756  0.175919  0.589821
          0.5           0.649476   0.108745  0.462219  0.176065  0.586259

In [202]:
save_result_df.groupby(
    ["mechanism", "method"]
)[["accuracy", "precision", "recall", "f1", "roc_auc"]].mean()

accuracy  precision    recall        f1   roc_auc
mechanism method                                                   
MAR       KNN     0.648777   0.108514  0.462165  0.175759  0.589627
          MICE    0.649298   0.109133  0.464737  0.176753  0.590476
          Mean    0.647544   0.109024  0.467095  0.176779  0.592478
          Median  0.646380   0.108932  0.468596  0.176767  0.592345
MCAR      KNN     0.645512   0.107489  0.462165  0.174409  0.590191
          MICE    0.652720   0.109267  0.459593  0.176553  0.588371
          Mean    0.651626   0.108455  0.457020  0.175307  0.589127
          Median  0.650132   0.109344  0.464309  0.176997  0.589812
MNAR      KNN     0.645078   0.108292  0.467310  0.175836  0.590432
          MICE    0.649211   0.108212  0.459807  0.175189  0.587211
          Mean    0.644835   0.108888  0.470954  0.176877  0.590217
          Median  0.648395   0.109174  0.466452  0.176932  0.588859